In [1]:
import numpy as np
import panel as pn
import plotly.graph_objects as go

pn.extension('plotly')

# ============================================================
# APP INTERACTIVA EN PANEL + PLOTLY
# KURAMOTO CLASICO PARA DOS OSCILADORES EN EL CIRCULO
# ============================================================
# Modelo:
#   dtheta1/dt = omega1 + (K/2) sin(theta2 - theta1)
#   dtheta2/dt = omega2 + (K/2) sin(theta1 - theta2)
#
# Los angulos iniciales se controlan en grados (0 a 360).
# La animacion se controla con un slider de tiempo, que es mucho
# mas estable que embeber animaciones pesadas en Jupyter.
# ============================================================


def deg2rad(deg):
    return deg * np.pi / 180.0


def rad2deg(rad):
    return rad * 180.0 / np.pi


def wrap_to_2pi(angle):
    return angle % (2 * np.pi)


def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def kuramoto_rhs(theta, omega1, omega2, K):
    theta1, theta2 = theta
    dtheta1 = omega1 + (K / 2.0) * np.sin(theta2 - theta1)
    dtheta2 = omega2 + (K / 2.0) * np.sin(theta1 - theta2)
    return np.array([dtheta1, dtheta2], dtype=float)


def simulate_kuramoto(theta1_0_deg, theta2_0_deg, omega1, omega2, K, T, dt):
    theta1_0 = deg2rad(theta1_0_deg)
    theta2_0 = deg2rad(theta2_0_deg)

    n_steps = int(T / dt) + 1
    t = np.linspace(0.0, T, n_steps)
    theta = np.zeros((n_steps, 2), dtype=float)
    theta[0] = [theta1_0, theta2_0]

    for k in range(n_steps - 1):
        y = theta[k]
        k1 = kuramoto_rhs(y, omega1, omega2, K)
        k2 = kuramoto_rhs(y + 0.5 * dt * k1, omega1, omega2, K)
        k3 = kuramoto_rhs(y + 0.5 * dt * k2, omega1, omega2, K)
        k4 = kuramoto_rhs(y + dt * k3, omega1, omega2, K)
        theta[k + 1] = y + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    theta_mod = wrap_to_2pi(theta)
    theta_deg_mod = rad2deg(theta_mod)
    theta_deg_unwrapped = rad2deg(theta)
    delta = np.array([wrap_to_pi(theta[k, 1] - theta[k, 0]) for k in range(len(t))])
    delta_deg = rad2deg(delta)
    R = np.abs(0.5 * (np.exp(1j * theta[:, 0]) + np.exp(1j * theta[:, 1])))

    return {
        't': t,
        'theta_deg_mod': theta_deg_mod,
        'theta_deg_unwrapped': theta_deg_unwrapped,
        'delta_deg': delta_deg,
        'R': R,
    }


def make_circle_figure(data, idx, trail):
    theta_deg = data['theta_deg_mod']
    t = data['t']

    th1 = deg2rad(theta_deg[:, 0])
    th2 = deg2rad(theta_deg[:, 1])
    x1, y1 = np.cos(th1), np.sin(th1)
    x2, y2 = np.cos(th2), np.sin(th2)

    start = max(0, idx - trail)

    circle = np.linspace(0, 2 * np.pi, 400)
    xc, yc = np.cos(circle), np.sin(circle)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xc, y=yc, mode='lines', name='Circulo'))

    fig.add_trace(go.Scatter(
        x=x1[start:idx+1], y=y1[start:idx+1],
        mode='lines', name='Trayectoria osc 1'
    ))
    fig.add_trace(go.Scatter(
        x=x2[start:idx+1], y=y2[start:idx+1],
        mode='lines', name='Trayectoria osc 2'
    ))

    fig.add_trace(go.Scatter(
        x=[0, x1[idx]], y=[0, y1[idx]], mode='lines',
        name='Radio osc 1', line=dict(dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=[0, x2[idx]], y=[0, y2[idx]], mode='lines',
        name='Radio osc 2', line=dict(dash='dash')
    ))

    fig.add_trace(go.Scatter(
        x=[x1[idx]], y=[y1[idx]], mode='markers', name='Osc 1',
        marker=dict(size=14)
    ))
    fig.add_trace(go.Scatter(
        x=[x2[idx]], y=[y2[idx]], mode='markers', name='Osc 2',
        marker=dict(size=14, symbol='square')
    ))

    fig.update_layout(
        title=f'Osciladores en el circulo | t = {t[idx]:.2f}',
        xaxis_title='x = cos(theta)',
        yaxis_title='y = sin(theta)',
        width=650,
        height=650,
        showlegend=True,
        xaxis=dict(scaleanchor='y', range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
        margin=dict(l=40, r=40, t=60, b=40),
    )
    return fig


def make_angles_figure(data, idx):
    t = data['t']
    th = data['theta_deg_unwrapped']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t[:idx+1], y=th[:idx+1, 0], mode='lines', name='theta1 (deg)'))
    fig.add_trace(go.Scatter(x=t[:idx+1], y=th[:idx+1, 1], mode='lines', name='theta2 (deg)'))
    fig.update_layout(
        title='Angulos acumulados',
        xaxis_title='t',
        yaxis_title='Grados',
        width=650,
        height=280,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


def make_sync_figure(data, idx):
    t = data['t']
    delta_deg = data['delta_deg']
    R = data['R']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t[:idx+1], y=delta_deg[:idx+1], mode='lines', name='Delta (deg)'))
    fig.add_trace(go.Scatter(x=t[:idx+1], y=R[:idx+1], mode='lines', name='R(t)', yaxis='y2'))

    fig.update_layout(
        title='Sincronizacion',
        xaxis_title='t',
        yaxis=dict(title='Delta (deg)'),
        yaxis2=dict(title='R(t)', overlaying='y', side='right', range=[-0.05, 1.05]),
        width=650,
        height=280,
        margin=dict(l=40, r=40, t=50, b=40),
    )
    return fig


# ------------------------------
# Widgets
# ------------------------------
theta1_widget = pn.widgets.FloatSlider(name='θ1(0) grados', start=0, end=360, step=1, value=30)
theta2_widget = pn.widgets.FloatSlider(name='θ2(0) grados', start=0, end=360, step=1, value=200)
omega1_widget = pn.widgets.FloatSlider(name='ω1', start=-5, end=5, step=0.1, value=1.0)
omega2_widget = pn.widgets.FloatSlider(name='ω2', start=-5, end=5, step=0.1, value=2.0)
K_widget = pn.widgets.FloatSlider(name='K', start=0, end=20, step=0.1, value=1.0)
T_widget = pn.widgets.FloatSlider(name='Tiempo total T', start=2, end=80, step=1, value=20)
dt_widget = pn.widgets.FloatSlider(name='dt', start=0.005, end=0.1, step=0.005, value=0.02)
trail_widget = pn.widgets.IntSlider(name='Cola visible', start=5, end=500, step=5, value=80)
play_widget = pn.widgets.Player(name='Tiempo', start=0, end=100, step=1, interval=80, value=0, loop_policy='loop')
status_md = pn.pane.Markdown('')


@pn.depends(theta1_widget, theta2_widget, omega1_widget, omega2_widget, K_widget, T_widget, dt_widget)
def compute_data(theta1_0, theta2_0, omega1, omega2, K, T, dt):
    return simulate_kuramoto(theta1_0, theta2_0, omega1, omega2, K, T, dt)


def bind_player_to_data(data):
    n = len(data['t'])
    play_widget.end = max(0, n - 1)
    if play_widget.value > play_widget.end:
        play_widget.value = play_widget.end


@pn.depends(theta1_widget, theta2_widget, omega1_widget, omega2_widget, K_widget, T_widget, dt_widget, trail_widget, play_widget)
def circle_panel(theta1_0, theta2_0, omega1, omega2, K, T, dt, trail, time_index):
    data = simulate_kuramoto(theta1_0, theta2_0, omega1, omega2, K, T, dt)
    bind_player_to_data(data)
    idx = min(time_index, len(data['t']) - 1)
    return pn.pane.Plotly(make_circle_figure(data, idx, trail), config={'responsive': True})


@pn.depends(theta1_widget, theta2_widget, omega1_widget, omega2_widget, K_widget, T_widget, dt_widget, play_widget)
def angles_panel(theta1_0, theta2_0, omega1, omega2, K, T, dt, time_index):
    data = simulate_kuramoto(theta1_0, theta2_0, omega1, omega2, K, T, dt)
    bind_player_to_data(data)
    idx = min(time_index, len(data['t']) - 1)
    return pn.pane.Plotly(make_angles_figure(data, idx), config={'responsive': True})


@pn.depends(theta1_widget, theta2_widget, omega1_widget, omega2_widget, K_widget, T_widget, dt_widget, play_widget)
def sync_panel(theta1_0, theta2_0, omega1, omega2, K, T, dt, time_index):
    data = simulate_kuramoto(theta1_0, theta2_0, omega1, omega2, K, T, dt)
    bind_player_to_data(data)
    idx = min(time_index, len(data['t']) - 1)
    return pn.pane.Plotly(make_sync_figure(data, idx), config={'responsive': True})


@pn.depends(theta1_widget, theta2_widget, omega1_widget, omega2_widget, K_widget, T_widget, dt_widget, play_widget)
def status_panel(theta1_0, theta2_0, omega1, omega2, K, T, dt, time_index):
    data = simulate_kuramoto(theta1_0, theta2_0, omega1, omega2, K, T, dt)
    bind_player_to_data(data)
    idx = min(time_index, len(data['t']) - 1)

    theta_mod = data['theta_deg_mod']
    delta_deg = data['delta_deg']
    R = data['R']
    t = data['t']

    return pn.pane.Markdown(
        f'''### Estado actual
- **t** = {t[idx]:.2f}
- **θ1(t)** = {theta_mod[idx, 0]:.2f}°
- **θ2(t)** = {theta_mod[idx, 1]:.2f}°
- **Δ(t)** = {delta_deg[idx]:.2f}°
- **R(t)** = {R[idx]:.4f}
- **ω1** = {omega1:.2f}
- **ω2** = {omega2:.2f}
- **K** = {K:.2f}
'''
    )


# Boton de reset
reset_btn = pn.widgets.Button(name='Restablecer parametros', button_type='primary')

# Funcion de reset
def reset_params(event=None):
    theta1_widget.value = 30
    theta2_widget.value = 200
    omega1_widget.value = 1.0
    omega2_widget.value = 2.0
    K_widget.value = 1.0
    T_widget.value = 20
    dt_widget.value = 0.02
    trail_widget.value = 80
    play_widget.value = 0

reset_btn.on_click(reset_params)

controls = pn.Card(
    'Mueve los parametros y usa el reproductor para avanzar en el tiempo.',
    theta1_widget,
    theta2_widget,
    omega1_widget,
    omega2_widget,
    K_widget,
    T_widget,
    dt_widget,
    trail_widget,
    play_widget,
    reset_btn,
    title='Controles',
    width=360,
)

main = pn.Column(
    pn.pane.Markdown('## Modelo de Kuramoto Clásico S^{1}'),
    pn.pane.Markdown(
        'Dos osciladores recorren el circulo. Si aumentas **ω1** o **ω2**, el punto correspondiente gira mas rapido. '
        'Si aumentas **K**, el acoplamiento empuja hacia la sincronizacion.'
    ),

    # 👇 CONTROLES ARRIBA
    controls,

    # 👇 CONTENIDO ABAJO
    pn.Row(
        pn.Column(status_panel, circle_panel),
        pn.Column(angles_panel, sync_panel)
    )
)

main.servable()

# Si lo corres en notebook, muestra la app con:
main


Column
    [0] Markdown(str)
    [1] Markdown(str)
    [2] Card(title='Controles', width=360)
        [0] Markdown(str)
        [1] FloatSlider(end=360, name='θ1(0) grados', step=1, value=30)
        [2] FloatSlider(end=360, name='θ2(0) grados', step=1, value=200)
        [3] FloatSlider(end=5, name='ω1', start=-5, value=1.0)
        [4] FloatSlider(end=5, name='ω2', start=-5, value=2.0)
        [5] FloatSlider(end=20, name='K', value=1.0)
        [6] FloatSlider(end=80, name='Tiempo total T', start=2, step=1, value=20)
        [7] FloatSlider(end=0.1, name='dt', start=0.005, step=0.005, value=0.02)
        [8] IntSlider(end=500, name='Cola visible', start=5, step=5, value=80)
        [9] Player(end=1000, interval=80, loop_policy='loop', name='Tiempo')
        [10] Button(button_type='primary', name='Restablecer parametros')
    [3] Row
        [0] Column
            [0] ParamFunction(function, _pane=Markdown, defer_load=False)
            [1] ParamFunction(function, _pane=Plotly, defer_load=False)
        [1] Column
            [0] ParamFunction(function, _pane=Plotly, defer_load=False)
            [1] ParamFunction(function, _pane=Plotly, defer_load=False)